# **Utility Functions**

This notebook is dedicated to writing the utility functions needed to pass to the optimiser from the `SatelliteSystem` object that handles the sceduling. Let us first create the satellite system,

In [1]:
# Importing,
import numpy as np
from Scheduler import SatelliteSystem, Station, Satellite

# Creating the system,
system = SatelliteSystem(planetary_radius=6000, planetary_mass=5.972e24, planetary_angular_velocity=7.2921159e-5)

# Adding ground stations,
station1 = Station(name="G1", position=np.pi/3)
station2 = Station(name="G2", position=2*np.pi/3)
station3 = Station(name="G3", position=4*np.pi/3)
system.add_station(station1)
system.add_station(station2)
system.add_station(station3)

# Adding satellites,
def create_ring(altitude, num_sats, label_prefix="SAT", orbit_prograde=True, transfer_rate=0, phase_offset=0):
    for i in range(num_sats):
        sat = Satellite(name=f"{label_prefix}-{i+1}", 
                        altitude=altitude, phase_angle=(2*np.pi*i/num_sats + phase_offset), 
                        orbit_prograde=orbit_prograde, 
                        transfer_rate=transfer_rate)
        system.add_sat(sat)

create_ring(altitude=500, num_sats=25, label_prefix="LEO-A", transfer_rate=250, phase_offset=np.pi/5, orbit_prograde=True)
create_ring(altitude=1000, num_sats=10, label_prefix="LEO-B", transfer_rate=100, phase_offset=np.pi/7, orbit_prograde=False)
create_ring(altitude=2000, num_sats=5, label_prefix="LEO-C", transfer_rate=25, phase_offset=np.pi/9, orbit_prograde=True)

Now we want to find the set $\mathcal{O}$ all indice tuples of overlapping windows,

In [19]:
def invervals_overlap(a, b):
    """It two time intervals a and overlap, the function returns True.
    Each interval is represented as a tuple (start, end)."""
    return a[1] > b[0] and a[0] < b[1]

# Calculating windows,
windows = system.compute_windows(station=station1, interval=(0, 24*60*60), format=False, order=True) # 24 hours in seconds.

# Finding overlapping windows,
O = []
for i in range(len(windows)):
    for j in range(i+1, len(windows)):
        a = windows[i][1:3]
        b = windows[j][1:3]
        if invervals_overlap(a, b):
            O.append((i, j))

print(O)

[(0, 1), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (3, 4), (3, 5), (3, 6), (3, 7), (3, 8), (4, 5), (4, 6), (4, 7), (4, 8), (4, 9), (4, 10), (4, 11), (4, 12), (4, 13), (4, 14), (4, 15), (4, 16), (4, 17), (5, 6), (5, 7), (6, 7), (6, 8), (6, 9), (6, 10), (7, 8), (7, 9), (7, 10), (7, 11), (8, 9), (8, 10), (8, 11), (9, 10), (9, 11), (9, 12), (10, 11), (10, 12), (10, 13), (10, 14), (10, 15), (11, 12), (11, 13), (11, 14), (12, 13), (12, 14), (12, 15), (13, 14), (13, 15), (13, 16), (13, 17), (13, 18), (14, 15), (14, 16), (14, 17), (14, 18), (14, 19), (15, 16), (15, 17), (15, 18), (15, 19), (16, 17), (16, 18), (16, 19), (16, 20), (16, 21), (16, 22), (16, 23), (16, 24), (16, 25), (16, 26), (16, 27), (16, 28), (17, 18), (17, 19), (17, 20), (18, 19), (18, 20), (18, 21), (18, 22), (18, 23), (19, 20), (19, 21), (19, 22), (20, 21), (20, 22), (20, 23), (21, 22), (21, 23), (21, 24), (22, 23), (22, 24), (22, 25), (22, 26), (23, 24), (23, 25), (23, 26), (24, 

Now we package this method,

In [ ]:
def compute_overlaps(windows):
    """Computes the overlapping windows from a list of windows. Each window is represented as a tuple 
    (satellite_name, start_time, end_time, duration, capacity). A list of tuples (i, j) is returned, 
    where i and j are the indices of the overlapping windows."""
    O = []
    for i in range(len(windows)):
        for j in range(i+1, len(windows)):
            a = windows[i][1:3]
            b = windows[j][1:3]
            if invervals_overlap(a, b):
                O.append((i, j))
    return O

### References

https://stackoverflow.com/questions/325933/determine-whether-two-date-ranges-overlap